# Default of Credit Card Clients — Data Generation (Multivariate)

this notebook loads the Default of Credit Card Clients dataset from the UCI repository, and generates three datasets with MCAR, MAR, and MNAR missingness using mdatagen.

run this notebook once to reproduce the CSV files used by `../analysis_multi.ipynb`. the CSVs are saved in the same `data/` folder as this notebook.

In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# initial data set loading
default_credit = fetch_ucirepo(id=350)
X_raw = default_credit.data.features
y = default_credit.data.targets.iloc[:, 0].to_numpy()

X_raw.isna().sum()

X_raw.shape

X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
dtype: int64

(30000, 23)

The dataset is already complete with no missing values.

We encode any categorical columns as integer codes so that every column is numeric — required by the mdatagen generators.

In [3]:
X = X_raw.copy()
cat_cols = X.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    X[col] = pd.factorize(X[col])[0]
X = X.astype(float)

X.isna().sum()

X.shape

X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
dtype: int64

(30000, 23)

Now we have a clean numeric dataset to work with. We inject missing values into multiple columns simultaneously to simulate realistic multivariate missingness patterns.

We use `default_payment_next_month` as the target variable `y` (0=no default, 1=default). It is passed to the generators because the library requires it — `default_payment_next_month` is not a feature in `X` and will not receive any missing values.

### MCAR generation

Here we'll generate a dataset with MCAR missingness across multiple columns simultaneously.

Missingness is completely random and does not depend on any observed variable or on `default_payment_next_month`.

In [4]:
np.random.seed(42)
df_mcar = mMCAR(X=X, y=y, missing_rate=30).random()

df_mcar.isna().sum()

df_mcar.shape

X1     9065
X2     9116
X3     8962
X4     8911
X5     8810
X6     9058
X7     8935
X8     8963
X9     8943
X10    9019
X11    8962
X12    8939
X13    9038
X14    9063
X15    8989
X16    9140
X17    8952
X18    9015
X19    9054
X20    9051
X21    9091
X22    8935
X23    8989
dtype: int64

(30000, 23)

### MAR generation

Here we'll generate a dataset with MAR missingness across multiple columns simultaneously.

Missingness is introduced using the random method from mdatagen. In `n_xmiss=15` randomly selected column pairs, the rows with the lowest values of the observed column are made missing in the other column — so missingness depends on observed values, not on the missing value itself.

In [5]:
np.random.seed(42)
df_mar = mMAR(X=X, y=y, n_xmiss=15).random(missing_rate=30)

df_mar.isna().sum()

df_mar.shape

X1        13800
X2            0
X3        13800
X4            0
X5        13800
X6        13800
X7            0
X8        13800
X9            0
X10       13800
X11           0
X12       13800
X13       13800
X14       13800
X15       13800
X16       13800
X17           0
X18       13800
X19       13800
X20           0
X21           0
X22       13800
X23       13800
target        0
dtype: int64

(30000, 24)

### MNAR generation

Here we'll generate a dataset with MNAR missingness across multiple columns simultaneously.

Missingness is generated on the prediction target `default_payment_next_month` using `threshold=1`. Rows at the lower end of the target distribution are most likely to have their feature records missing, simulating the case where certain client profiles are systematically underrepresented in the data.

Since `default_payment_next_month` is the target (`y`) and is absent from the generated feature matrix, its effect must be inferred through correlated proxies in the feature columns.

In [6]:
np.random.seed(42)
df_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=15).random(missing_rate=30)

df_mnar.isna().sum()

df_mnar.shape

X1            0
X2        13800
X3        13800
X4            0
X5        13800
X6            0
X7            0
X8        13800
X9        13800
X10       13800
X11       13800
X12       13800
X13           0
X14           0
X15           0
X16       13800
X17       13800
X18           0
X19       13800
X20       13800
X21       13800
X22       13800
X23       13800
target        0
dtype: int64

(30000, 24)

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

In [7]:
df_mcar.to_csv("mcar_multi.csv", index=False)
df_mar.to_csv("mar_multi.csv", index=False)
df_mnar.to_csv("mnar_multi.csv", index=False)